# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Azar Miguel Augusto**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

Por simplicidad lo ejecuto en **Colab** y para eso clono el repo en esta celda

In [1]:
!git clone https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/intro_ia.git
%cd intro_ia/clase2/content/hanoi_tower

Cloning into 'intro_ia'...
remote: Enumerating objects: 3056, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 3056 (delta 137), reused 164 (delta 121), pack-reused 2826 (from 3)
Receiving objects: 100% (3056/3056), 631.94 MiB | 31.19 MiB/s, done.
Resolving deltas: 100% (1195/1195), done.
Updating files: 100% (313/313), done.
/content/intro_ia/clase2/content/hanoi_tower


In [2]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

## Heurística elegida A*

Implemento **Búsqueda A\*** usando una heurística propia simple:

- `h(n)` cuenta cuántos discos **todavía no están bien ubicados** en la varilla destino (varilla 3), empezando a contar desde la base.

Un disco se considera bien ubicado si el y todos los discos más grandes que deberían ir debajo de el en la varilla destino ya están ahí, en el orden correcto. En cuanto aparece el primer disco que no coincide con el estado objetivo, se deja de contar (porque ese disco, y todos los que faltan, van a tener que moverse sí o sí).

por lo tanto, `h(n) = cantidad_total_de_discos - discos_bien_ubicados`.

- Si ningún disco está en su lugar final: `h(n) = numero_discos` (el máximo).
- Si todos los discos están en su lugar final (estado objetivo): `h(n) = 0`.



In [3]:
import heapq
import itertools


def heuristic(state: StatesHanoi, goal: StatesHanoi) -> float:
    """
    Cuenta los discos que aun NO estan bien ubicados en la varilla destino.

    Recorremos la varilla destino (indice 2) desde la base (indice 0) hacia
    arriba, comparando el estado actual contra el estado objetivo. Contamos
    cuantos discos coinciden en forma consecutiva desde la base. Apenas
    encontramos un disco que no coincide, paramos de contar: ese disco (y
    todos los que le siguen) todavia tienen que moverse.

    h(n) = cantidad_total_de_discos - discos_bien_ubicados
    """
    target_rod_state = state.rods[2]
    target_rod_goal = goal.rods[2]

    well_placed = 0
    for disk_state, disk_goal in zip(target_rod_state, target_rod_goal):
        if disk_state == disk_goal:
            well_placed += 1
        else:
            break

    return state.number_of_disks - well_placed

In [4]:
def search_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA

    root = NodeHanoi(initial_state)

    # La frontera es una cola de prioridad ordenada por f(n) = g(n) + h(n).
    # 'counter' es un desempatador: si dos nodos tienen el mismo f(n), heapq
    # necesita comparar el siguiente elemento de la tupla, y no queremos que
    # intente comparar objetos NodeHanoi entre si.
    counter = itertools.count()
    frontier = []
    h_root = heuristic(initial_state, goal_state)
    heapq.heappush(frontier, (root.path_cost + h_root, next(counter), root))

    explored = set()  # Estados ya visitados
    nodes_explored = 0
    max_depth = 0

    while frontier:
        _, _, node = heapq.heappop(frontier)  # heapq.heappop siempre devuelve
                                              # el elemento con el valor más
                                              # chico de la lista frontier

        # Puede haber quedado una version vieja (peor) del mismo estado en
        # la frontera. Si ya lo expandimos, la saltamos.
        if node.state in explored:
            continue # ya fue explorado

        explored.add(node.state)
        nodes_explored += 1
        max_depth = max(max_depth, node.depth)

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": nodes_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        for child in node.expand(problem):
            if child.state not in explored:
                h_child = heuristic(child.state, goal_state)
                heapq.heappush(
                    frontier, (child.path_cost + h_child, next(counter), child)
                )

    # no se encontro solucion
    metrics = {
        "solution_found": False,
        "nodes_explored": nodes_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": None,
    }
    return None, metrics
    #####

Se prueba la función:

In [5]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [6]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 169
states_visited: 169
nodes_in_frontier: 18
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [7]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [8]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
